# Task 2: Inference & Evaluation — BERT

**Model:** `task2-m4pro-bert-lr3e5-bs32-ep3-window_1`  
**Metrics:** MRR · Hits@1/3/5

In [1]:
import json
from pathlib import Path

MODEL_DIR   = "./outputs/models/task2-m4pro-bert-lr3e5-bs32-ep3-window_1"
TEST_DIR    = "./data_outputs/task2/test"
RESULTS_DIR = "./outputs/results"
BATCH_SIZE  = 32

RUN_NAME     = Path(MODEL_DIR).name
cfg          = json.loads(Path(f"./outputs/configs/{RUN_NAME}.json").read_text())
CONTEXT_MODE = cfg["context_mode"]
MAX_LENGTH   = cfg["max_length"]

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
print(f"✅ Model   : {RUN_NAME}")
print(f"   context : {CONTEXT_MODE}")
print(f"   max_len : {MAX_LENGTH}")

✅ Model   : task2-m4pro-bert-lr3e5-bs32-ep3-window_1
   context : window_1
   max_len : 512


In [2]:
import json, re, csv, os
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("mps" if torch.backends.mps.is_available() else
                      "cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

/Users/tathiyennhi/Documents/Works/automatic-citation-checking/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Device: mps


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()
print(f"✅ Model loaded")

✅ Model loaded


In [4]:
def get_context(text, citation_id, mode="window_1"):
    if mode == "full":
        return text
    window = int(mode.split("_")[1])
    sentences = re.split(r"(?<=[.!?])\s+", text)
    target_idx = next((i for i, s in enumerate(sentences) if citation_id in s), -1)
    if target_idx == -1:
        return text
    start = max(0, target_idx - window)
    end   = min(len(sentences), target_idx + window + 1)
    return " ".join(sentences[start:end])


def load_ranking_groups(test_dir, context_mode):
    label_files = sorted(Path(test_dir).glob("*.label"))
    total = len(label_files)
    print(f"📊 Loading {total:,} test files...")

    groups, skipped = [], 0
    for idx, lf in enumerate(label_files):
        if (idx + 1) % 500 == 0:
            print(f"  ⏳ {idx+1:,}/{total:,}")
        try:
            in_data    = json.loads(lf.with_suffix(".in").read_text())
            label_data = json.loads(lf.read_text())
        except Exception:
            skipped += 1; continue

        text        = in_data.get("text", "")
        candidates  = in_data.get("citation_candidates", [])
        bib_entries = in_data.get("bib_entries", {})
        correct_map = label_data.get("correct_citation", {})

        if not text or not candidates or not bib_entries or not correct_map:
            skipped += 1; continue

        for citation_id, correct_paper_id in correct_map.items():
            context = get_context(text, citation_id, mode=context_mode)
            cands = [
                {
                    "paper_id":   pid,
                    "text_b":     "{}.{}".format(
                        bib_entries[pid].get("title", ""),
                        bib_entries[pid].get("abstract", "")
                    ),
                    "is_correct": pid == correct_paper_id,
                }
                for pid in candidates if pid in bib_entries
            ]
            if cands:
                groups.append({"context": context, "candidates": cands})

    print(f"\n✅ {len(groups):,} ranking groups | skipped: {skipped}")
    return groups


groups = load_ranking_groups(TEST_DIR, CONTEXT_MODE)

📊 Loading 3,000 test files...
  ⏳ 500/3,000
  ⏳ 1,000/3,000
  ⏳ 1,500/3,000
  ⏳ 2,000/3,000
  ⏳ 2,500/3,000
  ⏳ 3,000/3,000

✅ 7,151 ranking groups | skipped: 0


In [5]:
texts_a, texts_b, group_ids = [], [], []
for gi, g in enumerate(groups):
    for c in g["candidates"]:
        texts_a.append(g["context"])
        texts_b.append(c["text_b"])
        group_ids.append(gi)

total_pairs = len(texts_a)
print(f"Total pairs to score: {total_pairs:,} ({len(groups):,} groups)")

all_scores = np.zeros(total_pairs, dtype=np.float32)

with torch.no_grad():
    for start in range(0, total_pairs, BATCH_SIZE):
        if start % (BATCH_SIZE * 100) == 0:
            print(f"  ⏳ {start:,}/{total_pairs:,}")
        end = min(start + BATCH_SIZE, total_pairs)
        enc = tokenizer(
            texts_a[start:end], texts_b[start:end],
            max_length=MAX_LENGTH, truncation=True,
            padding="max_length", return_tensors="pt"
        ).to(device)
        logits = model(**enc).logits
        scores = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_scores[start:end] = scores

score_cursor = 0
for gi, g in enumerate(groups):
    n = len(g["candidates"])
    g["scores"] = all_scores[score_cursor: score_cursor + n].tolist()
    score_cursor += n

print(f"\n✅ Scoring done")

Total pairs to score: 150,838 (7,151 groups)
  ⏳ 0/150,838
  ⏳ 3,200/150,838
  ⏳ 6,400/150,838
  ⏳ 9,600/150,838
  ⏳ 12,800/150,838
  ⏳ 16,000/150,838
  ⏳ 19,200/150,838
  ⏳ 22,400/150,838
  ⏳ 25,600/150,838
  ⏳ 28,800/150,838
  ⏳ 32,000/150,838
  ⏳ 35,200/150,838
  ⏳ 38,400/150,838
  ⏳ 41,600/150,838
  ⏳ 44,800/150,838
  ⏳ 48,000/150,838
  ⏳ 51,200/150,838
  ⏳ 54,400/150,838
  ⏳ 57,600/150,838
  ⏳ 60,800/150,838
  ⏳ 64,000/150,838
  ⏳ 67,200/150,838
  ⏳ 70,400/150,838
  ⏳ 73,600/150,838
  ⏳ 76,800/150,838
  ⏳ 80,000/150,838
  ⏳ 83,200/150,838
  ⏳ 86,400/150,838
  ⏳ 89,600/150,838
  ⏳ 92,800/150,838
  ⏳ 96,000/150,838
  ⏳ 99,200/150,838
  ⏳ 102,400/150,838
  ⏳ 105,600/150,838
  ⏳ 108,800/150,838
  ⏳ 112,000/150,838
  ⏳ 115,200/150,838
  ⏳ 118,400/150,838
  ⏳ 121,600/150,838
  ⏳ 124,800/150,838
  ⏳ 128,000/150,838
  ⏳ 131,200/150,838
  ⏳ 134,400/150,838
  ⏳ 137,600/150,838
  ⏳ 140,800/150,838
  ⏳ 144,000/150,838
  ⏳ 147,200/150,838
  ⏳ 150,400/150,838

✅ Scoring done


In [6]:
reciprocal_ranks = []
hits = {1: 0, 3: 0, 5: 0}

for g in groups:
    ranked_idx = np.argsort(g["scores"])[::-1]
    correct_ranks = [
        rank + 1
        for rank, idx in enumerate(ranked_idx)
        if g["candidates"][idx]["is_correct"]
    ]
    if not correct_ranks:
        reciprocal_ranks.append(0.0)
        continue
    best = min(correct_ranks)
    reciprocal_ranks.append(1.0 / best)
    for k in hits:
        if best <= k:
            hits[k] += 1

n   = len(groups)
mrr = float(np.mean(reciprocal_ranks))

print("=" * 55)
print(f"🎯 RANKING METRICS | {RUN_NAME}")
print("=" * 55)
print(f"  Queries : {n:,}")
print(f"  MRR     : {mrr:.4f}")
for k in [1, 3, 5]:
    print(f"  Hits@{k}  : {hits[k]/n:.4f}  ({hits[k]:,}/{n:,})")
print("=" * 55)

🎯 RANKING METRICS | task2-m4pro-bert-lr3e5-bs32-ep3-window_1
  Queries : 7,151
  MRR     : 0.5618
  Hits@1  : 0.3914  (2,799/7,151)
  Hits@3  : 0.6669  (4,769/7,151)
  Hits@5  : 0.7831  (5,600/7,151)


In [7]:
test_results = {
    "run_name":      RUN_NAME,
    "n_queries":     n,
    "n_total_pairs": int(total_pairs),
    "mrr":           round(mrr, 4),
    "hits_at_1":     round(hits[1] / n, 4),
    "hits_at_3":     round(hits[3] / n, 4),
    "hits_at_5":     round(hits[5] / n, 4),
}

json_path = f"{RESULTS_DIR}/{RUN_NAME}_test_results.json"
with open(json_path, "w") as f:
    json.dump(test_results, f, indent=2)
print(f"✅ JSON: {json_path}")

csv_path = f"{RESULTS_DIR}/test_summary.csv"
write_hdr = not os.path.exists(csv_path)
with open(csv_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=test_results.keys())
    if write_hdr:
        writer.writeheader()
    writer.writerow(test_results)
print(f"✅ CSV : {csv_path}")

✅ JSON: ./outputs/results/task2-m4pro-bert-lr3e5-bs32-ep3-window_1_test_results.json
✅ CSV : ./outputs/results/test_summary.csv
